In [1]:
!pip install openpyxl

import openpyxl

from datetime import datetime
from openpyxl.styles import Border, Side
from openpyxl import load_workbook


In [2]:
# ------------------------------------------------------------------------------売上データを作成
# 新しいワークブックを作成
wb = openpyxl.Workbook()
ws = wb.active

# シート名を変更しておく
ws.title = "売上"

# 表のヘッダー行を書き込む
header =  ['商品名', '数量', '単価', '金額']
target_ws = wb["売上"]
target_ws.append(header)

# 表のデータを書き込む
data = [
    ['商品A', 2, 10000, 20000],
    ['商品B', 1, 15000, 15000]
]
# for文でdataの各行をワークシートに追加する
for row in data:
    ws.append(row)

# セルA1:D4に上下左右に枠線を追加
thin_border = Border(left=Side(style='thin'),
                    right=Side(style='thin'),
                    top=Side(style='thin'),
                    bottom=Side(style='thin'))
# セル範囲をsheet['A1:D3'] と指定して、for文で各セルに枠線を追加
for row in ws['A1:D3']:
    for cell in row:
        cell.border = thin_border

# ------------------------------------------------------------------------------ファイルを保存

wb.save('売上データ.xlsx')


In [5]:
# ------------------------------------------------------------------------------請求書を作成
# 売上データ を読み込む
wb_uriage = openpyxl.load_workbook('売上データ.xlsx')
ws_uriage = wb_uriage["売上"]
# ws_uriage = wb_uriage.active

#  新しいワークブックを作成する
wb_seikyuu = openpyxl.Workbook()
ws_seikyuu = wb_seikyuu.active
ws_seikyuu.title = "請求書"

# 初期化
sales_data = []
uri_total = 0
syo_total = 0
intax = 0
# 税率は10％
tax_rate = 10

# 売上データからsales_data リストを作成する
for row in ws_uriage.iter_rows(min_row=2, values_only=True):
  item = [row[0], row[1], row[2], row[3]]
  sales_data.append(item)

#------------------------------------------------------シート作成
# -------------------------定数のセット
ws_seikyuu['B2'] = "請求書"
ws_seikyuu['B4'] = "株式会社ABC"
ws_seikyuu['B5'] = "〒101-0022 東京都千代田区神田練塀町300"
ws_seikyuu['B6'] = "TEL:03-1234-5678 FAX:03-1234-5678"
ws_seikyuu['B7'] = "担当者名:鈴木一郎 様"

ws_seikyuu['F4'] = "№"
ws_seikyuu['F5'] = "日付"
ws_seikyuu['G4'] = "0001"

ws_seikyuu['B10'] = "商品名"
ws_seikyuu['C10'] = "数量"
ws_seikyuu['D10'] = "単価"
ws_seikyuu['E10'] = "金額"

ws_seikyuu['B15'] = "小計"
ws_seikyuu['B16'] = "消費税"
ws_seikyuu['B17'] = "税込合計"
#  日付
today = datetime.today()
form_today = today.strftime('%Y/%m/%d')
ws_seikyuu['G5'] = form_today

# -------------------------明細エリア
start_row = 11    # 開始行セット

# sales_data リストから1商品ずつ取り出す
for i, item in enumerate(sales_data):
    row = start_row + i

    ws_seikyuu.cell(row=row, column=2, value=item[0])
    ws_seikyuu.cell(row=row, column=3, value=item[1])
    ws_seikyuu.cell(row=row, column=4, value=item[2])
    ws_seikyuu.cell(row=row, column=5, value=item[3])

    uri_total += int(item[3])
    syo_total += int(item[3])

#  -------------------------集計エリア
ws_seikyuu['E13'] = uri_total
ws_seikyuu['E15'] = syo_total
tax = syo_total * tax_rate / 100
ws_seikyuu['E16'] = tax
intax = uri_total + tax
ws_seikyuu['E17'] = intax

# 住所が日付に干渉するため、列幅調整
for col in ['B', 'C', 'D', 'E']:
    ws_seikyuu.column_dimensions[col].width = 10

# ファイル名
backup_filename = f"請求書_{today.strftime('%Y%m%d')}.xlsx"
wb_seikyuu.save(backup_filename)
